<!-- Banner Image -->
<center>
    <img src="https://developer-blogs.nvidia.com/wp-content/uploads/2024/07/rag-representation.jpg" width="75%">
</center>

<!-- Links -->
<center>
  <a href="https://www.nvidia.com/en-us/deep-learning-ai/solutions/data-science/workbench/" style="color: #76B900;">NVIDIA AI Workbench</a> •
  <a href="https://docs.nvidia.com/ai-workbench/" style="color: #76B900;">User Documentation</a> •
  <a href="https://docs.nvidia.com/ai-workbench/user-guide/latest/quickstart/example-projects.html" style="color: #76B900;">Example Projects Catalog</a> •
  <a href="https://forums.developer.nvidia.com/t/support-workbench-example-project-llama-3-finetune/303411" style="color: #76B900;"> Problem? Submit a ticket here! </a>
</center>

# Finetune Llama-3.3-8B-Instruct on DGX Spark GB10 using SFT

Welcome!

This notebook is optimized for **NVIDIA DGX Spark with GB10** (Grace Blackwell architecture). With 128GB of unified memory, we can perform **full-precision BF16 training** without any quantization.

### Key Features for DGX Spark GB10:
- **No quantization needed** - 128GB unified memory handles full BF16 training easily
- **Native SDPA attention** - Uses PyTorch's Scaled Dot Product Attention (no Flash Attention required)
- **Larger batch sizes** - More memory allows for faster training
- **Full model fine-tuning** - No need for LoRA/QLoRA memory optimizations

#### Help us make this tutorial better! Please provide feedback on the [NVIDIA Developer Forum](https://forums.developer.nvidia.com/c/ai-data-science/nvidia-ai-workbench/671).

A note about running Jupyter Notebooks: Press Shift + Enter to run a cell. A * in the left-hand cell box means the cell is running. A number means it has completed.

## Table of Contents
1. Verify GPU and Environment
2. Import libraries
3. Load model and dataset
4. Configure training
5. Train the model
6. Deploy as an OpenAI compatible endpoint

## 1. Verify GPU and Environment

Let's first verify that we're running on the DGX Spark GB10 and that PyTorch can see the GPU.

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")
else:
    print("WARNING: CUDA not available. Please check your installation.")

## 2. Import Libraries

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2024 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    logging,
)
from trl import SFTTrainer

## 3. Load Llama 3.3 and Dataset

We load the model in **BF16 precision** without any quantization. The DGX Spark GB10's 128GB unified memory easily handles this.

**Important:** We use `attn_implementation="sdpa"` (Scaled Dot Product Attention) instead of Flash Attention, as SDPA is natively supported on Blackwell architecture and is very fast.

In [ ]:
# Model and dataset configuration
base_model_id = "allura-forge/Llama-3.3-8B-Instruct"
dataset_name = "scooterman/guanaco-llama3-1k"
new_model = "/project/models/NV-llama3.3-8b-SFT"

In [ ]:
# Load dataset
dataset = load_dataset(dataset_name, split="train")
print(f"Dataset loaded: {len(dataset)} samples")

In [ ]:
# Load model in BF16 - NO QUANTIZATION needed on DGX Spark GB10!
# Using SDPA (Scaled Dot Product Attention) instead of Flash Attention for Blackwell compatibility
model = AutoModelForCausalLM.from_pretrained(
    base_model_id, 
    token=os.environ["HF_KEY"], 
    cache_dir="/project/models",
    torch_dtype=torch.bfloat16,  # Full BF16 precision
    attn_implementation="sdpa",   # Use native SDPA instead of Flash Attention
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    token=os.environ["HF_KEY"], 
    add_eos_token=True,
    add_bos_token=True, 
)
tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded in BF16 with SDPA attention")
print(f"Model parameters: {model.num_parameters():,}")

## 4. Configure Training

With 128GB of unified memory on the DGX Spark GB10, we can use **larger batch sizes** for faster training. The settings below are optimized for the GB10's capabilities.

In [ ]:
# Output directory where the results and checkpoint are stored
output_dir = "./results"

# Number of training epochs
num_train_epochs = 1

# Use BF16 training - optimal for Blackwell architecture
bf16 = True

# DGX Spark GB10 optimized batch size - can be larger due to 128GB memory
per_device_train_batch_size = 4  # Increased from 1 for GB10

# Gradient accumulation steps
gradient_accumulation_steps = 4  # Reduced since we have larger batch size

# Gradient checkpointing - can disable on GB10 for faster training if memory allows
gradient_checkpointing = False  # Disabled for speed on GB10

# Maximum gradient norm (gradient clipping)
max_grad_norm = 0.3

# Initial learning rate
learning_rate = 2e-4

# Weight decay
weight_decay = 0.001

# Optimizer - use standard AdamW on GB10 (no need for paged optimizer)
optim = "adamw_torch"

# Number of training steps
max_steps = 500

# Warmup ratio
warmup_ratio = 0.03

# Group sequences by length for efficiency
group_by_length = True

# Save and logging steps
save_steps = 100
logging_steps = 5

## (Optional) Enable Weights & Biases Logging

Uncomment the cells below to enable W&B logging for experiment tracking.

In [ ]:
### Uncomment to use Weights and Biases ###

# import wandb
# wandb.login()

In [ ]:
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    gradient_checkpointing=gradient_checkpointing,
    report_to="none",  # Change to "wandb" if using W&B
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_arguments,
)

## 5. Train the Model

In [ ]:
trainer.train()

## Save the Model

Save the fine-tuned model and tokenizer to the `/project/models` directory.

In [ ]:
trainer.model.save_pretrained(new_model)
trainer.tokenizer.save_pretrained(new_model)
print(f"Model saved to: {new_model}")

## 6. Deploy as an OpenAI-Compatible Endpoint

Deploy the fine-tuned model using vLLM. Open a new terminal to test the endpoint.

In [ ]:
!python3 -m vllm.entrypoints.openai.api_server \
    --host=0.0.0.0 \
    --port=8000 \
    --model=/project/models/NV-llama3.3-8b-SFT \
    --tokenizer=allura-forge/Llama-3.3-8B-Instruct \
    --dtype=bfloat16 \
    --tensor-parallel-size=1

# Test the endpoint from a new terminal:
# curl http://localhost:8000/v1/completions \
#    -H "Content-Type: application/json" \
#    -d '{
#        "model": "/project/models/NV-llama3.3-8b-SFT",
#        "prompt": "What is San Francisco",
#        "max_tokens": 30,
#        "temperature": 0
#    }'